In [1]:
# Parámetros a configurar:
import os, sys, json, time, random, re, shutil, subprocess, itertools
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("C:/Users/Teresa/Documents/GitHub/navsim")
experiment_name = "command15"
task_name = "Command15"
cfg_dir         = ROOT / "isaac_lab/aerotaxi/env_command/env_19/grid_hiperparametros/cfg_files"
path_resultados = ROOT / "tmp/tmpTeresa/resultados_grid_command_v19"
path_ral        = ROOT / "tmp/tmpTeresa/resultados_ral_v19"
isaaclab_bat    = "C:/Users/Teresa/Desktop/RuralData/IsaacLab/IsaacLab/isaaclab.bat"
train_script    = str(ROOT / "isaac_lab/rl_v5_1_0/rsl_rl/train.py")
play_script     = str(ROOT / "isaac_lab/rl_v5_1_0/rsl_rl/play.py")
id_task         = f"Isaac-{task_name}-Aerotaxi-RANDOM"
 
NUM_ENVS_TRAIN    = 8192               
NUM_STEPS_PER_ENV = 128                
NUM_ITER          = 4000               
NUM_ITER_STAGE2   = 1500               
N_CONFIGS         = 35
GRID_SEED         = 19                 
REUSE_ENV17_CFGS  = False              
ENV17_CFG_DIR     = ROOT / "isaac_lab/aerotaxi/env_command/env_17/grid_hiperparametros/cfg_files"
 
TRAIN_SEEDS  = [11, 22, 33, 44, 55]    
TOP_K        = 3                       
EVAL_SEED    = 1001                    
N_TEST_ENVS  = 1024
DT           = 0.02                    
BURN_IN_S    = 5.0                     
 
for d in (cfg_dir, path_resultados, path_ral, path_resultados / "logs", path_ral / "logs"):
    d.mkdir(parents=True, exist_ok=True)

# el num_steps_per_env = 100 también está en el env, al final del todo, y deben tener el mismo valor
# Recuerda iniciar en el init todos los experimentos

In [2]:
# Opciones de hiperparámetros
param_options = {
    "init_noise_std": [0.1, 0.2, 0.5, 0.8, 1.0],
    "actor_hidden_dims": [[64, 64], [128, 128], [256, 256], [128, 64]],
    "critic_hidden_dims": [[64, 64], [128, 128], [256, 256]],
    "activation": ["relu", "tanh", "elu", "gelu"],
    "value_loss_coef": [0.5, 1.0, 2.0],
    "clip_param": [0.1, 0.2, 0.3],
    "entropy_coef": [0.0, 0.001, 0.01, 0.05],
    "num_learning_epochs": [4, 8, 12],
    "num_mini_batches": [2, 4, 8],
    "learning_rate": [5e-05, 1e-4, 3e-4, 1e-3, 3e-3],
    "gamma": [0.95, 0.98, 0.99],
    "lam": [0.90, 0.95, 0.97],
    "max_grad_norm": [0.5, 1.0, 2.0],
    "desired_kl": [0.005, 0.008, 0.01, 0.02],
}

CFG_TEMPLATE = '''from isaaclab.utils import configclass
from isaaclab_rl.rsl_rl import RslRlOnPolicyRunnerCfg, RslRlPpoActorCriticCfg, RslRlPpoAlgorithmCfg
 
@configclass
class {task_name}PPORunnerCfg(RslRlOnPolicyRunnerCfg):
    seed = {seed}
    num_steps_per_env = {num_steps}
    max_iterations = {num_iter}
    save_interval = 100
    experiment_name = "{experiment_name}"
    run_name = "{run_name}"
    resume = False
    empirical_normalization = True
    csv_path_metrics = "{csv_path}"
    callbacks = True
    early_stopping_patience = {patience}      
    policy = RslRlPpoActorCriticCfg(
        init_noise_std={init_noise_std},
        actor_hidden_dims={actor_hidden_dims},
        critic_hidden_dims={critic_hidden_dims},
        activation="{activation}",
    )
    algorithm = RslRlPpoAlgorithmCfg(
        value_loss_coef={value_loss_coef},
        use_clipped_value_loss=True,
        clip_param={clip_param},
        entropy_coef={entropy_coef},
        num_learning_epochs={num_learning_epochs},
        num_mini_batches={num_mini_batches},
        learning_rate={learning_rate},
        schedule="adaptive",
        gamma={gamma},
        lam={lam},
        desired_kl={desired_kl},
        max_grad_norm={max_grad_norm},
    )
'''

def write_cfg(index: int, hp: dict, seed: int, csv_path: Path = path_resultados, num_iter: int | None = None,
              patience: int = 500) -> str:
    run_name = f"ppo_random_{experiment_name}_{index}"
    body = CFG_TEMPLATE.format(task_name=task_name, seed=seed, num_steps=NUM_STEPS_PER_ENV, num_iter=num_iter or NUM_ITER,
                               experiment_name=experiment_name, run_name=run_name,
                               csv_path=str(csv_path).replace("\\", "/"), patience=patience, **hp)
    (cfg_dir / f"{run_name}_cfg.py").write_text(body, encoding="utf-8")
    return run_name
 
manifest_path = cfg_dir / "grid_manifest.json"
if manifest_path.exists():
    print("grid_manifest.json ya existe -> NO se regenera (evita sobrescribir el diseño del experimento).")
    grid = json.loads(manifest_path.read_text())
else:
    rng = random.Random(GRID_SEED)
    grid = {}
    for i in range(1, N_CONFIGS + 1):
        hp = {k: rng.choice(v) for k, v in param_options.items()}
        if REUSE_ENV17_CFGS:                                  # mismos hiperparámetros que env_17
            old = (ENV17_CFG_DIR / f"ppo_random_command13_{i}_cfg.py").read_text(encoding="utf-8")
            pick = lambda pat, cast=float: cast(re.search(pat, old).group(1))
            hp = dict(
                init_noise_std=pick(r"init_noise_std=([\d.e-]+)"),
                actor_hidden_dims=json.loads(re.search(r"actor_hidden_dims=(\[.*?\])", old).group(1)),
                critic_hidden_dims=json.loads(re.search(r"critic_hidden_dims=(\[.*?\])", old).group(1)),
                activation=re.search(r'activation="(\w+)"', old).group(1),
                value_loss_coef=pick(r"value_loss_coef=([\d.e-]+)"), clip_param=pick(r"clip_param=([\d.e-]+)"),
                entropy_coef=pick(r"entropy_coef=([\d.e-]+)"), num_learning_epochs=pick(r"num_learning_epochs=(\d+)", int),
                num_mini_batches=pick(r"num_mini_batches=(\d+)", int), learning_rate=pick(r"learning_rate=([\d.e-]+)"),
                gamma=pick(r"gamma=([\d.e-]+)"), lam=pick(r"lam=([\d.e-]+)"),
                desired_kl=pick(r"desired_kl=([\d.e-]+)"), max_grad_norm=pick(r"max_grad_norm=([\d.e-]+)"))
        grid[str(i)] = {"hp": hp, "seed": 42}                  # etapa 1: UNA semilla común
        write_cfg(i, hp, seed=42)
    manifest_path.write_text(json.dumps(grid, indent=1))
print(f"{len(grid)} configuraciones en {cfg_dir}")

grid_manifest.json ya existe -> NO se regenera (evita sobrescribir el diseño del experimento).
35 configuraciones en C:\Users\Teresa\Documents\GitHub\navsim\isaac_lab\aerotaxi\env_command\env_19\grid_hiperparametros\cfg_files


In [3]:
registry_csv = path_resultados / "run_registry.csv"
 
def git_state():
    sh = lambda c: subprocess.run(c, cwd=ROOT, capture_output=True, text=True).stdout.strip()
    return {"git_commit": sh(["git", "rev-parse", "HEAD"]), "git_dirty": bool(sh(["git", "status", "--porcelain"]))}
 
def run_cmd(args, log_path: Path, env_extra: dict | None = None, timeout: int | None = None):
    env = os.environ.copy()
    for k in ("AEROTAXI_TAU_MS", "AEROTAXI_WIND_MS", "AEROTAXI_GUST_STD", "AEROTAXI_TRAJ", "HORIZON_MODE"):
        env.pop(k, None)                                    # nunca heredar overrides de una celda anterior
    env.update({k: str(v) for k, v in (env_extra or {}).items()})
    log_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with open(log_path, "w", encoding="utf-8") as f:
        try:
            rc = subprocess.run([str(a) for a in args], stdout=f, stderr=subprocess.STDOUT, env=env, timeout=timeout).returncode
        except subprocess.TimeoutExpired:
            rc = -9
    return rc, time.time() - t0
 
def register(row: dict):
    row = {**row, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **git_state()}
    pd.DataFrame([row]).to_csv(registry_csv, mode="a", header=not registry_csv.exists(), index=False)
 
def train_one(index: int, seed: int, env_extra: dict | None = None, tag: str = ""):
    """OK solo si: rc==0 Y sin 'Traceback' en el log Y existe el checkpoint del runner.
    (isaaclab.bat puede devolver rc=0 aunque train.py haya lanzado una excepción.)"""
    ok = path_resultados / "done" / f"train_{index}{tag}.ok"
    if ok.exists():
        return "skipped"
    run = f"ppo_random_{experiment_name}_{index}"
    log = path_resultados / "logs" / f"train_{index}{tag}.log"
    args = [isaaclab_bat, "-p", train_script, "--task", f"{id_task}-{index}",
            "--num_envs", NUM_ENVS_TRAIN, "--seed", seed, "--headless"]
    rc, dt = run_cmd(args, log, env_extra)
    text = log.read_text(encoding="utf-8", errors="ignore")
    has_tb = "Traceback (most recent call last)" in text
    has_ckpt = any(list(d.glob(f"best_model_{run}_*.pt")) for d in (path_resultados, path_ral))
    good = (rc == 0) and not has_tb and has_ckpt
    register({"phase": "train", "index": index, "seed": seed, "tag": tag, "rc": rc, "traceback": has_tb,
              "checkpoint": has_ckpt, "seconds": round(dt), **{f"env_{k}": v for k, v in (env_extra or {}).items()}})
    if good:
        ok.parent.mkdir(exist_ok=True); ok.write_text("ok")
        return 0
    print(f"\n--- FALLO config {index} (rc={rc}, traceback={has_tb}, checkpoint={has_ckpt}) · últimas líneas de {log.name}:")
    print("\n".join(text.splitlines()[-25:]))
    return rc or 1

In [ ]:
smoke_dir = path_resultados / "_smoke"
write_cfg(100, grid["1"]["hp"], seed=1, csv_path=smoke_dir, num_iter=3, patience=0)
log = path_resultados / "logs" / "smoke.log"
rc, _ = run_cmd([isaaclab_bat, "-p", train_script, "--task", f"{id_task}-100", "--num_envs", 64,
                 "--max_iterations", 3, "--seed", 1, "--headless"], log)
txt = log.read_text(encoding="utf-8", errors="ignore")
print("rc =", rc, "| traceback:", "Traceback" in txt)
print("\n".join(txt.splitlines()[-25:]))

rc = 0 | traceback: False
             Mean action noise std: 0.11
          Mean value_function loss: 0.6380
               Mean surrogate loss: 0.0057
                 Mean entropy loss: -3.1366
                       Mean reward: 8.26
               Mean episode length: 51.16
Episode_Reward/rew_attitude_stability2: 0.0460
Episode_Reward/rew_ang_vel_stability4: 0.0236
           Episode_Reward/rew_pos2: 0.0668
    Episode_Reward/rew_action_rate: -0.0000
        Episode_Reward/rew_heading: 0.0383
     Episode_Reward/rew_track_vel3: 0.1182
       Metrics/vel_command/pos_mse: 0.5213
       Metrics/vel_command/vel_mse: 3.5501
       Metrics/vel_command/yaw_abs: 1.5933
      Episode_Termination/time_out: 0.0000
Episode_Termination/below_min_altitude: 0.0110
  Episode_Termination/bad_attitude: 0.9890
Episode_Termination/safety_shutdown: 0.0000
--------------------------------------------------------------------------------
                   Total timesteps: 24576
                    Itera

In [ ]:
ck = next(smoke_dir.glob("final_model_*_64.pt"))
log = path_resultados / "logs" / "smoke_play.log"
rc, _ = run_cmd([isaaclab_bat, "-p", play_script, "--task", f"{id_task}-100", "--num_envs", 64, "--checkpoint", ck,
                 "--headless", "--seed", 7, "--tag", "smoke", "--record_states", "--states_csv_dir", smoke_dir / "traj"],
                log, env_extra={"AEROTAXI_TAU_MS": 50})
txt = log.read_text(encoding="utf-8", errors="ignore")
print("rc =", rc, "| traceback:", "Traceback" in txt); print("\n".join(txt.splitlines()[-12:]))
row = pd.read_csv(smoke_dir / "rewards_play_PPO.csv").iloc[-1]
print(row[["tag", "n_episodes", "success", "aerotaxi_tau_ms"]])
ep = episode_metrics(pd.read_csv(next((smoke_dir / "traj").glob("tracking_3D_*.csv"))))
assert row.n_episodes == 64 and row.aerotaxi_tau_ms == 50 and len(ep) == 64

In [ ]:
for i in range(1, N_CONFIGS + 1):
    rc = train_one(i, seed=grid[str(i)]["seed"])
    print(f"config {i:>2}: {'OK' if rc in (0, 'skipped') else f'FALLO rc={rc}'}")   # un fallo ya no aborta el resto

In [ ]:
df_best = pd.read_csv(path_resultados / "reward_best_models.csv")
df_best["success_train"] = df_best["mean_Episode_Termination/time_out"]
df_best["cfg"] = df_best["id_run_name"].str.extract(r"_(\d+)$").astype(int)
shortlist = (df_best[df_best["success_train"] > 0.85]
             .sort_values("reward", ascending=False).head(TOP_K)["cfg"].tolist())
print("Configs que pasan a etapa 2:", shortlist)
display(df_best.sort_values("reward", ascending=False)[["cfg", "reward", "std_reward", "success_train", "iteration"]].head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Teresa\\Documents\\GitHub\\navsim\\tmp\\tmpTeresa\\resultados_grid_command_v19\\reward_best_models.csv'

In [ ]:
stage2 = {}                                     # index -> (cfg_base, seed)
idx = 36
for base in shortlist:
    for s in TRAIN_SEEDS:
        write_cfg(idx, grid[str(base)]["hp"], seed=s, csv_path=path_ral, num_iter=NUM_ITER_STAGE2, patience=0)
        stage2[idx] = (base, s); idx += 1
assert idx - 1 <= 100, "el __init__ solo registra 1..100"
json.dump({str(k): v for k, v in stage2.items()}, open(cfg_dir / "stage2_manifest.json", "w"))
for i, (base, s) in stage2.items():
    print(i, base, s, train_one(i, seed=s))

In [ ]:
best_cfg = shortlist[0]
abl = {}
for mode in ("fixed", "none"):
    for s in TRAIN_SEEDS:
        write_cfg(idx, grid[str(best_cfg)]["hp"], seed=s, csv_path=path_ral, num_iter=NUM_ITER_STAGE2, patience=0)
        abl[idx] = (best_cfg, s, mode); idx += 1
assert idx - 1 <= 100
for i, (base, s, mode) in abl.items():
    print(i, mode, s, train_one(i, seed=s, env_extra={"HORIZON_MODE": mode}, tag=f"_{mode}"))

In [ ]:
CONDITIONS = (
    [dict(name="nominal", traj="random", tau_ms=0, wind_ms=None)]                                  # distribución de entrenamiento
    + [dict(name=f"tau{t}", traj="random", tau_ms=t, wind_ms=None) for t in (25, 50, 75, 100)]       # >40 ms = fuera de rango
    + [dict(name=f"wind{w}", traj="random", tau_ms=0, wind_ms=w) for w in (0, 12, 16)]              # 12-16 > rango de train (0-8)
    + [dict(name=t, traj=t, tau_ms=0, wind_ms=None) for t in ("circle", "lemniscate")]               # trayectorias no vistas
)
 
def eval_one(index, seed_train, horizon, cond, ckpt: Path):
    tag = f"idx{index}_s{seed_train}_{horizon}_{cond['name']}"
    out = path_ral / "eval" / tag
    if list(out.glob("tracking_3D_*.csv")):
        return "skipped"
    env_extra = {"AEROTAXI_TRAJ": cond["traj"], "HORIZON_MODE": horizon}
    if cond["tau_ms"] is not None: env_extra["AEROTAXI_TAU_MS"] = cond["tau_ms"]
    if cond["wind_ms"] is not None: env_extra["AEROTAXI_WIND_MS"] = cond["wind_ms"]; env_extra["AEROTAXI_GUST_STD"] = 0 if cond["wind_ms"] == 0 else 2.0
    args = [isaaclab_bat, "-p", play_script, "--task", f"{id_task}-{index}", "--num_envs", N_TEST_ENVS,
            "--checkpoint", ckpt, "--headless", "--seed", EVAL_SEED, "--tag", tag,
            "--record_states", "--states_csv_dir", out]
    rc, dt = run_cmd(args, path_ral / "logs" / f"play_{tag}.log", env_extra)
    register({"phase": "eval", "index": index, "seed_train": seed_train, "horizon": horizon, "cond": cond["name"],
              "seed_eval": EVAL_SEED, "rc": rc, "seconds": round(dt), **{f"env_{k}": v for k, v in env_extra.items()}})
    return rc
 
def ckpt_of(index):    # runner parcheado: final_model_{run_name}_{num_envs}.pt (modelo de la ÚLTIMA iteración, sin selección por reward ruidoso)
    return next(path_ral.glob(f"final_model_ppo_random_{experiment_name}_{index}_*.pt"))
 
policies = [(i, s, "stochastic", b) for i, (b, s) in stage2.items()] + [(i, s, m, b) for i, (b, s, m) in abl.items()]
for (i, s, h, b) in policies:
    conds = CONDITIONS if h == "stochastic" else CONDITIONS[:1]            # ablación: solo condición nominal
    for c in conds:
        print(i, h, c["name"], eval_one(i, s, h, c, ckpt_of(i)))

In [ ]:
def episode_metrics(df: pd.DataFrame, dt=DT, burn_in_s=BURN_IN_S) -> pd.DataFrame:
    need = {"env_id", "episode", "step", "drone_x", "drone_y", "drone_z", "target_x", "target_y", "target_z", "terminated"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en el CSV de trayectorias (revisa PLAY_CONTRACT): {sorted(missing)}")
    d = df.sort_values(["env_id", "episode", "step"]).copy()
    d["err"] = np.sqrt((d.drone_x - d.target_x) ** 2 + (d.drone_y - d.target_y) ** 2 + (d.drone_z - d.target_z) ** 2)
    d["t"] = d.groupby(["env_id", "episode"]).cumcount() * dt
    g = d.groupby(["env_id", "episode"])
    ep = g.agg(T=("t", "max"), failed=("terminated", "max"), rmse_all=("err", lambda e: float(np.sqrt(np.mean(e ** 2)))),
               emax=("err", "max")).reset_index()
    ss = d[d.t >= burn_in_s].groupby(["env_id", "episode"])["err"].apply(lambda e: float(np.sqrt(np.mean(e ** 2)))).rename("rmse_ss")
    ep = ep.merge(ss.reset_index(), on=["env_id", "episode"], how="left")
    ep["failed"] = ep["failed"].astype(bool)
    ep["success"] = ~ep["failed"]
    ep["time_to_failure"] = np.where(ep.failed, ep["T"], np.nan)
    return ep
 
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return (c - h, c + h)
 
def boot_ci(x, stat=np.mean, n=10000, alpha=0.05, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    s = [stat(rng.choice(x, len(x), replace=True)) for _ in range(n)]
    return tuple(np.quantile(s, [alpha / 2, 1 - alpha / 2]))
 
def iqm(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return float(x[int(np.floor(.25 * n)): int(np.ceil(.75 * n))].mean()) if n >= 4 else float(np.mean(x))
 
def load_all_episodes() -> pd.DataFrame:
    rows = []
    for csv in (path_ral / "eval").glob("*/tracking_3D_*.csv"):
        m = re.match(r"idx(\d+)_s(\d+)_(\w+?)_(\w+)$", csv.parent.name)
        index, seed_train, horizon, cond = int(m[1]), int(m[2]), m[3], m[4]
        ep = episode_metrics(pd.read_csv(csv))
        ep["index"], ep["seed_train"], ep["horizon"], ep["cond"] = index, seed_train, horizon, cond
        ep["cfg"] = (stage2.get(index) or abl.get(index))[0]
        rows.append(ep)
    return pd.concat(rows, ignore_index=True)

In [ ]:
def per_policy(ep: pd.DataFrame) -> pd.DataFrame:
    return (ep.groupby(["cfg", "horizon", "cond", "seed_train"])
              .agg(success=("success", "mean"), rmse_ss=("rmse_ss", "median"), rmse_all=("rmse_all", "median"),
                   ttf=("time_to_failure", "median"), n_ep=("success", "size")).reset_index())
 
def summarize(pp: pd.DataFrame, by=("cfg", "horizon", "cond")) -> pd.DataFrame:
    out = []
    for key, g in pp.groupby(list(by)):
        lo_s, hi_s = boot_ci(g.success); lo_r, hi_r = boot_ci(g.rmse_ss)
        out.append(dict(zip(by, key)) | dict(n_seeds=len(g), success_pct=100 * g.success.mean(),
                        success_ci=f"[{100*lo_s:.1f}, {100*hi_s:.1f}]", rmse_ss_m=g.rmse_ss.mean(),
                        rmse_ss_ci=f"[{lo_r:.2f}, {hi_r:.2f}]", iqm_rmse_ss=iqm(g.rmse_ss)))
    return pd.DataFrame(out)
 
def paired_delta(pp, cond_a, cond_b, metric="rmse_ss"):
    """Diferencia pareada por semilla de entrenamiento (mismas políticas, distinta condición) + IC bootstrap."""
    a = pp[pp.cond == cond_a].set_index(["cfg", "horizon", "seed_train"])[metric]
    b = pp[pp.cond == cond_b].set_index(["cfg", "horizon", "seed_train"])[metric]
    d = (b - a).dropna()
    return d.mean(), boot_ci(d)
 
episodes = load_all_episodes()
pp = per_policy(episodes)
tab = summarize(pp)
display(tab)
print("tau 50 ms - nominal, RMSE_ss:", paired_delta(pp, "nominal", "tau50"))
print("tau 50 ms - nominal, success :", paired_delta(pp, "nominal", "tau50", "success"))
 
def to_latex(df, path, caption, label):
    Path(path).write_text(df.to_latex(index=False, escape=True, float_format="%.2f", caption=caption, label=label,
                                      column_format="l" * len(df.columns), position="t"), encoding="utf-8")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
OKABE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # apto para daltonismo
def ieee_style():
    mpl.rcParams.update({"font.size": 8, "font.family": "serif", "axes.labelsize": 8, "legend.fontsize": 7,
                         "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.prop_cycle": mpl.cycler(color=OKABE),
                         "pdf.fonttype": 42, "ps.fonttype": 42, "axes.grid": True, "grid.alpha": .3, "grid.linestyle": ":",
                         "savefig.bbox": "tight", "savefig.dpi": 300})
ieee_style()
 
def curves(df_log: pd.DataFrame, run_names) -> pd.DataFrame:
    """Recorta a la iteración común mínima (evita el artefacto de media sobre n variable de semillas)."""
    cut = min(df_log[df_log.id_run_name == r].iteration.max() for r in run_names)
    wide = pd.concat([df_log[(df_log.id_run_name == r) & (df_log.iteration <= cut)]
                      .drop_duplicates("iteration").set_index("iteration").reward.rename(r) for r in run_names], axis=1)
    return wide
 
def plot_band(ax, wide, label, color, ls="-"):
    m = wide.mean(axis=1); lo = wide.quantile(0.025, axis=1) if wide.shape[1] > 5 else wide.min(axis=1)
    hi = wide.quantile(0.975, axis=1) if wide.shape[1] > 5 else wide.max(axis=1)
    ax.plot(m.index, m, color=color, ls=ls, lw=1.2, label=f"{label} (n={wide.shape[1]})")
    ax.fill_between(m.index, lo, hi, color=color, alpha=.2, lw=0)
 
log_ral = pd.read_csv(path_ral / "resultados_grid_PPO.csv")
fig, ax = plt.subplots(1, 2, figsize=(7.16, 2.3))
arms = {"stochastic": [f"ppo_random_{experiment_name}_{i}" for i, (b, s) in stage2.items() if b == best_cfg],
        "fixed": [f"ppo_random_{experiment_name}_{i}" for i, (b, s, m) in abl.items() if m == "fixed"],
        "none":  [f"ppo_random_{experiment_name}_{i}" for i, (b, s, m) in abl.items() if m == "none"]}
for (name, runs), c, ls in zip(arms.items(), OKABE, ("-", "--", "-.")):
    plot_band(ax[0], curves(log_ral, runs), {"stochastic": "Stochastic [0.4, 0.9] s", "fixed": "Fixed 0.65 s", "none": "None"}[name], c, ls)
ax[0].set_xlabel("PPO iteration"); ax[0].set_ylabel("Training episode return"); ax[0].legend(loc="lower right")
# panel derecha: robustez a retardo del actuador (RMSE_ss y éxito, media ± IC sobre semillas)
taus = [0, 25, 50, 75, 100]
sub = pp[(pp.horizon == "stochastic") & pp.cond.isin(["nominal"] + [f"tau{t}" for t in taus[1:]])]
sub = sub.assign(tau=sub.cond.map(lambda c: 0 if c == "nominal" else int(c[3:])))
g = sub.groupby("tau")
ax[1].errorbar(g.rmse_ss.mean().index, g.rmse_ss.mean(), yerr=[g.rmse_ss.mean() - g.rmse_ss.apply(lambda x: boot_ci(x)[0]),
               g.rmse_ss.apply(lambda x: boot_ci(x)[1]) - g.rmse_ss.mean()], marker="o", ms=3, capsize=2, color=OKABE[0])
ax[1].set_xlabel("Actuator time constant $\\tau$ (ms)"); ax[1].set_ylabel("Steady-state RMSE (m)", color=OKABE[0])
ax1b = ax[1].twinx(); ax1b.grid(False)
ax1b.errorbar(g.success.mean().index, 100 * g.success.mean(), yerr=None, marker="s", ms=3, color=OKABE[1])
ax1b.set_ylabel("Success rate (%)", color=OKABE[1]); ax[1].axvspan(40, 100, color="grey", alpha=.08, lw=0)   # fuera del rango de entrenamiento
fig.tight_layout(); fig.savefig(path_ral / "fig_convergence_ablation_latency.pdf"); plt.show()

In [ ]:
def pip_freeze_subset():
    out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
    return [l for l in out if re.match(r"(torch|numpy|pandas|isaaclab|rsl|gymnasium|scipy)", l, re.I)]
 
manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), **git_state(), "task": id_task, "env": "env_19",
            "num_envs_train": NUM_ENVS_TRAIN, "num_steps_per_env": NUM_STEPS_PER_ENV, "max_iterations": NUM_ITER,
            "train_seeds": TRAIN_SEEDS, "eval_seed": EVAL_SEED, "n_test_envs": N_TEST_ENVS, "grid_seed": GRID_SEED,
            "shortlist": shortlist, "conditions": CONDITIONS, "packages": pip_freeze_subset(),
            "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                                  capture_output=True, text=True).stdout.strip()}
(path_ral / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str))
print(json.dumps({k: manifest[k] for k in ("git_commit", "git_dirty", "gpu")}, indent=1))

In [ ]:
import json, re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# ---------- 1) persistencia de estado ----------
def save_state(path_ral, **kw):
    (path_ral / "state.json").write_text(json.dumps(kw, indent=1, default=lambda o: list(o)))

def load_state(path_ral):
    st = json.loads((path_ral / "state.json").read_text())
    stage2 = {int(k): tuple(v) for k, v in st["stage2"].items()}
    abl    = {int(k): tuple(v) for k, v in st["abl"].items()}
    nodr   = {int(k): tuple(v) for k, v in st.get("nodr", {}).items()}
    return st["shortlist"], st["best_cfg"], stage2, abl, nodr

# ---------- 2) tablas ----------
def make_tables(pp, summarize, out_dir):
    tab = summarize(pp)
    conds = ["nominal", "tau25", "tau50", "tau75", "tau100", "wind0", "wind12", "wind16", "circle", "lemniscate"]
    t = tab[tab.cond.isin(conds) & tab.horizon.isin(["stochastic", "nodr"])].copy()
    t["Success (\\%)"] = t.success_pct.map("{:.1f}".format) + " " + t.success_ci
    t["RMSE$_{ss}$ (m)"] = t.rmse_ss_m.map("{:.2f}".format) + " " + t.rmse_ss_ci
    piv = t.pivot_table(index="cond", columns="horizon", values=["Success (\\%)", "RMSE$_{ss}$ (m)"], aggfunc="first")
    piv = piv.reindex([c for c in conds if c in piv.index])
    (Path(out_dir) / "table_robustness.tex").write_text(
        piv.to_latex(escape=False, caption="Success rate and steady-state RMSE (mean, 95\\% bootstrap CI over training seeds).",
                     label="tab:robustness"), encoding="utf-8")
    a = tab[tab.cond == "nominal"].copy()
    (Path(out_dir) / "table_horizon_ablation.tex").write_text(
        a[["horizon", "n_seeds", "success_pct", "success_ci", "rmse_ss_m", "rmse_ss_ci"]].to_latex(
            index=False, float_format="%.2f", caption="Look-ahead horizon ablation (nominal conditions).", label="tab:horizon"),
        encoding="utf-8")
    return piv

# ---------- 3) figura de trayectorias (episodio MEDIANO, no el mejor) ----------
def traj_fig(path_ral, episode_metrics, index, seed_train, horizon, conds=("nominal", "circle", "lemniscate"), name="fig_trajectories.pdf"):
    fig, axs = plt.subplots(1, len(conds), figsize=(7.16, 2.3))
    for ax, c in zip(np.atleast_1d(axs), conds):
        d = pd.read_csv(next((path_ral / "eval" / f"idx{index}_s{seed_train}_{horizon}_{c}").glob("tracking_3D_*.csv")))
        ep = episode_metrics(d).dropna(subset=["rmse_ss"])
        env = ep.loc[(ep.rmse_ss - ep.rmse_ss.median()).abs().idxmin(), "env_id"]
        g = d[d.env_id == env]
        ax.plot(g.target_x, g.target_y, "k--", lw=1, label="Reference")
        ax.plot(g.drone_x, g.drone_y, lw=1.2, color="#0072B2", label="PPO")
        ax.set_title(c, fontsize=8); ax.set_aspect("equal", adjustable="datalim"); ax.set_xlabel("x (m)")
    np.atleast_1d(axs)[0].set_ylabel("y (m)"); np.atleast_1d(axs)[0].legend(loc="best")
    fig.tight_layout(); fig.savefig(path_ral / name); plt.close(fig)